# Composition Over Inheritance – "Has-a" Relationships for Real-World Modeling

Welcome to the fourteenth notebook in our Object-Oriented Programming (OOP) for Computer Vision series! In this tutorial, we'll explore the principle of composition over inheritance, a powerful design approach that favors object composition over class inheritance. We'll see how this principle can be applied to create more flexible, maintainable, and realistic models for computer vision applications.

## Table of Contents
1. [Introduction to Composition Over Inheritance](#introduction)
2. [Inheritance vs. Composition](#inheritance-vs-composition)
3. [When to Use Composition](#when-to-use-composition)
4. [Implementing Composition in Python](#implementing-composition)
5. [Delegation Pattern](#delegation-pattern)
6. [Composition in Computer Vision](#cv-applications)
7. [Best Practices](#best-practices)
8. [Exercises](#exercises)
9. [Conclusion](#conclusion)

## 1. Introduction to Composition Over Inheritance <a name="introduction"></a>

"Composition over inheritance" is a design principle that suggests that classes should achieve polymorphic behavior and code reuse by their composition (by containing instances of other classes that implement the desired functionality) rather than through inheritance (by inheriting from a parent class or implementing an interface).

This principle is often summarized as "favor 'has-a' relationships over 'is-a' relationships."

### Key Benefits of Composition Over Inheritance

1. **Flexibility**: Composition allows for more flexible designs that can be changed at runtime.
2. **Reduced Coupling**: Composition reduces the coupling between classes, making the code more maintainable.
3. **Avoids Fragile Base Class Problem**: Composition avoids the "fragile base class problem," where changes to a base class can unexpectedly break derived classes.
4. **Simplifies Testing**: Composition makes it easier to test classes in isolation by mocking or stubbing their dependencies.
5. **Promotes Single Responsibility**: Composition encourages classes to have a single responsibility, improving code organization.

Let's import the necessary libraries for our examples:

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional, Any, Union, Callable
import os
from abc import ABC, abstractmethod

# Helper function to display images
def display_image(image, title="Image"):
    plt.figure(figsize=(8, 6))
    if len(image.shape) == 3:  # Color image
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    else:  # Grayscale image
        plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()
    
# Create a sample image (a simple gradient with shapes)
def create_sample_image(width=300, height=200):
    # Create a gradient from black to white
    gradient = np.zeros((height, width), dtype=np.uint8)
    for i in range(width):
        gradient[:, i] = int(255 * i / width)
    
    # Convert to a color image (BGR)
    color_gradient = cv2.cvtColor(gradient, cv2.COLOR_GRAY2BGR)
    
    # Add some shapes
    cv2.circle(color_gradient, (width//4, height//2), 40, (0, 0, 255), -1)  # Red circle
    cv2.rectangle(color_gradient, (width//2, height//4), (3*width//4, 3*height//4), (0, 255, 0), -1)  # Green rectangle
    cv2.line(color_gradient, (0, 0), (width, height), (255, 0, 0), 5)  # Blue line
    
    return color_gradient

# Create a sample image
sample_image = create_sample_image()

# Display the sample image
display_image(sample_image, "Sample Image")

## 2. Inheritance vs. Composition <a name="inheritance-vs-composition"></a>

Before diving deeper into composition, let's compare inheritance and composition to understand their differences and when to use each.

### Inheritance: "Is-a" Relationship

Inheritance represents an "is-a" relationship between classes. For example, a `GrayscaleProcessor` is a type of `ImageProcessor`.

```python
class ImageProcessor:
    def process(self, image):
        return image

class GrayscaleProcessor(ImageProcessor):
    def process(self, image):
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
```

### Composition: "Has-a" Relationship

Composition represents a "has-a" relationship between classes. For example, an `ImageProcessor` has a `filter` that it applies to images.

```python
class Filter:
    def apply(self, image):
        return image

class GrayscaleFilter(Filter):
    def apply(self, image):
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

class ImageProcessor:
    def __init__(self, filter):
        self.filter = filter
    
    def process(self, image):
        return self.filter.apply(image)
```

### Key Differences

1. **Relationship Type**: Inheritance represents an "is-a" relationship, while composition represents a "has-a" relationship.
2. **Flexibility**: Composition is more flexible because you can change the behavior at runtime by changing the composed objects.
3. **Coupling**: Inheritance creates tight coupling between classes, while composition creates loose coupling.
4. **Code Reuse**: Inheritance reuses code by inheriting from a parent class, while composition reuses code by containing instances of other classes.
5. **Extensibility**: Composition is often more extensible because you can add new behaviors without modifying existing classes.

Let's see a more concrete example of both approaches:

In [ ]:
# Inheritance approach
class ImageProcessor:
    def __init__(self, name="Base Processor"):
        self.name = name
    
    def process(self, image):
        """Process an image (base implementation)."""
        print(f"{self.name}: Processing image...")
        return image
    
    def display_result(self, image):
        """Display the original and processed images side by side."""
        processed_image = self.process(image)
        
        plt.figure(figsize=(12, 6))
        
        # Display original image
        plt.subplot(1, 2, 1)
        if len(image.shape) == 3:  # Color image
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        else:  # Grayscale image
            plt.imshow(image, cmap='gray')
        plt.title("Original Image")
        plt.axis('off')
        
        # Display processed image
        plt.subplot(1, 2, 2)
        if len(processed_image.shape) == 3:  # Color image
            plt.imshow(cv2.cvtColor(processed_image, cv2.COLOR_BGR2RGB))
        else:  # Grayscale image
            plt.imshow(processed_image, cmap='gray')
        plt.title(f"{self.name} Result")
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return processed_image

class GrayscaleProcessor(ImageProcessor):
    def __init__(self):
        super().__init__("Grayscale Processor")
    
    def process(self, image):
        """Convert the image to grayscale."""
        print(f"{self.name}: Converting image to grayscale...")
        if len(image.shape) == 2 or image.shape[2] == 1:
            return image  # Already grayscale
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

class BlurProcessor(ImageProcessor):
    def __init__(self, kernel_size=(5, 5)):
        super().__init__(f"Blur Processor ({kernel_size})")
        self.kernel_size = kernel_size
    
    def process(self, image):
        """Apply Gaussian blur to the image."""
        print(f"{self.name}: Applying Gaussian blur...")
        return cv2.GaussianBlur(image, self.kernel_size, 0)

# Composition approach
class Filter:
    def __init__(self, name="Base Filter"):
        self.name = name
    
    def apply(self, image):
        """Apply the filter to an image (base implementation)."""
        print(f"{self.name}: Applying filter...")
        return image

class GrayscaleFilter(Filter):
    def __init__(self):
        super().__init__("Grayscale Filter")
    
    def apply(self, image):
        """Convert the image to grayscale."""
        print(f"{self.name}: Converting image to grayscale...")
        if len(image.shape) == 2 or image.shape[2] == 1:
            return image  # Already grayscale
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

class BlurFilter(Filter):
    def __init__(self, kernel_size=(5, 5)):
        super().__init__(f"Blur Filter ({kernel_size})")
        self.kernel_size = kernel_size
    
    def apply(self, image):
        """Apply Gaussian blur to the image."""
        print(f"{self.name}: Applying Gaussian blur...")
        return cv2.GaussianBlur(image, self.kernel_size, 0)

class ImageProcessorComposition:
    def __init__(self, filter, name="Composition Processor"):
        self.filter = filter
        self.name = name
    
    def process(self, image):
        """Process an image using the filter."""
        print(f"{self.name}: Processing image using {self.filter.name}...")
        return self.filter.apply(image)
    
    def display_result(self, image):
        """Display the original and processed images side by side."""
        processed_image = self.process(image)
        
        plt.figure(figsize=(12, 6))
        
        # Display original image
        plt.subplot(1, 2, 1)
        if len(image.shape) == 3:  # Color image
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        else:  # Grayscale image
            plt.imshow(image, cmap='gray')
        plt.title("Original Image")
        plt.axis('off')
        
        # Display processed image
        plt.subplot(1, 2, 2)
        if len(processed_image.shape) == 3:  # Color image
            plt.imshow(cv2.cvtColor(processed_image, cv2.COLOR_BGR2RGB))
        else:  # Grayscale image
            plt.imshow(processed_image, cmap='gray')
        plt.title(f"{self.name} Result\nUsing {self.filter.name}")
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return processed_image

# Create instances of the inheritance-based processors
grayscale_processor = GrayscaleProcessor()
blur_processor = BlurProcessor(kernel_size=(15, 15))

# Create instances of the composition-based processors
grayscale_filter = GrayscaleFilter()
blur_filter = BlurFilter(kernel_size=(15, 15))
grayscale_processor_comp = ImageProcessorComposition(grayscale_filter)
blur_processor_comp = ImageProcessorComposition(blur_filter)

# Process and display the results using inheritance-based processors
print("Using inheritance-based processors:")
grayscale_processor.display_result(sample_image)
blur_processor.display_result(sample_image)

# Process and display the results using composition-based processors
print("\nUsing composition-based processors:")
grayscale_processor_comp.display_result(sample_image)
blur_processor_comp.display_result(sample_image)

In this example, we've implemented the same functionality using both inheritance and composition. The inheritance approach creates a hierarchy of `ImageProcessor` classes, with each subclass overriding the `process` method to provide specific behavior. The composition approach separates the filtering logic into a hierarchy of `Filter` classes, and then composes an `ImageProcessorComposition` class that uses a filter to process images.

Both approaches achieve the same result, but the composition approach has several advantages:

1. **Flexibility**: We can change the filter at runtime, allowing for more dynamic behavior.
2. **Separation of Concerns**: The filtering logic is separated from the image processing logic, making the code more modular.
3. **Extensibility**: We can add new filters without modifying the `ImageProcessorComposition` class.

Let's see how the composition approach allows for more flexible behavior:

In [ ]:
# Create a new filter
class EdgeFilter(Filter):
    def __init__(self, threshold1=100, threshold2=200):
        super().__init__(f"Edge Filter ({threshold1}, {threshold2})")
        self.threshold1 = threshold1
        self.threshold2 = threshold2
    
    def apply(self, image):
        """Detect edges in the image using Canny edge detection."""
        print(f"{self.name}: Detecting edges...")
        
        # Convert to grayscale if needed
        if len(image.shape) == 3 and image.shape[2] > 1:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image
        
        return cv2.Canny(gray, self.threshold1, self.threshold2)

# Create a new processor with the edge filter
edge_filter = EdgeFilter(threshold1=50, threshold2=150)
edge_processor = ImageProcessorComposition(edge_filter)

# Process and display the result
edge_processor.display_result(sample_image)

# Change the filter at runtime
edge_processor.filter = blur_filter
edge_processor.display_result(sample_image)

In this example, we've created a new filter `EdgeFilter` and used it to create a new processor `edge_processor`. We've then changed the filter at runtime to use the `blur_filter` instead. This demonstrates the flexibility of the composition approach, allowing us to change the behavior of an object at runtime without creating new classes or modifying existing ones.

## 3. When to Use Composition <a name="when-to-use-composition"></a>

While composition offers many advantages, it's not always the best choice. Here are some guidelines for when to use composition over inheritance:

### Use Composition When:

1. **You Need Runtime Flexibility**: If you need to change behavior at runtime, composition is more flexible.
2. **You Want to Avoid the Fragile Base Class Problem**: If changes to a base class could break derived classes, composition reduces this risk.
3. **You Need Multiple Behaviors**: If a class needs to exhibit multiple behaviors that don't fit neatly into an inheritance hierarchy, composition allows for more flexibility.
4. **You Want to Favor Delegation Over Inheritance**: If you want to delegate specific tasks to specialized classes, composition is a better fit.
5. **You Want to Avoid Deep Inheritance Hierarchies**: Deep inheritance hierarchies can be hard to understand and maintain; composition can flatten these hierarchies.

### Use Inheritance When:

1. **There's a Clear "Is-a" Relationship**: If there's a clear "is-a" relationship between classes (e.g., a `Circle` is a `Shape`), inheritance may be more appropriate.
2. **You Want to Share Code Among Related Classes**: If you have common code that should be shared among related classes, inheritance can be a good way to achieve this.
3. **You Need Polymorphic Behavior**: If you need objects of different classes to be treated as objects of a common superclass, inheritance provides this through polymorphism.
4. **You're Implementing an Interface**: If you're implementing an interface or abstract base class, inheritance is necessary.

In practice, a combination of inheritance and composition is often the best approach. Use inheritance for "is-a" relationships and to share common code, and use composition for "has-a" relationships and to achieve runtime flexibility.

## 4. Implementing Composition in Python <a name="implementing-composition"></a>

Python makes it easy to implement composition through its flexible object model. Here are some common patterns for implementing composition in Python:

### Basic Composition

The most basic form of composition is to include instances of other classes as attributes of a class:

In [ ]:
class Engine:
    def __init__(self, power):
        self.power = power
    
    def start(self):
        print(f"Engine starting with {self.power} horsepower...")
    
    def stop(self):
        print("Engine stopping...")

class Wheels:
    def __init__(self, count):
        self.count = count
    
    def rotate(self):
        print(f"{self.count} wheels rotating...")
    
    def stop(self):
        print(f"{self.count} wheels stopping...")

class Car:
    def __init__(self, engine, wheels):
        self.engine = engine
        self.wheels = wheels
    
    def start(self):
        self.engine.start()
    
    def stop(self):
        self.engine.stop()
        self.wheels.stop()
    
    def drive(self):
        print("Car driving...")
        self.wheels.rotate()

# Create a car with an engine and wheels
engine = Engine(power=200)
wheels = Wheels(count=4)
car = Car(engine=engine, wheels=wheels)

# Use the car
car.start()
car.drive()
car.stop()

In this example, the `Car` class is composed of an `Engine` and `Wheels`. The `Car` delegates specific tasks to these components, such as starting the engine or rotating the wheels.

### Composition with Factory Methods

Sometimes, you might want to create the composed objects within the class itself, rather than passing them in from outside. This can be done using factory methods:

In [ ]:
class Car:
    def __init__(self, engine_power=150, wheel_count=4):
        self.engine = self._create_engine(engine_power)
        self.wheels = self._create_wheels(wheel_count)
    
    def _create_engine(self, power):
        return Engine(power)
    
    def _create_wheels(self, count):
        return Wheels(count)
    
    def start(self):
        self.engine.start()
    
    def stop(self):
        self.engine.stop()
        self.wheels.stop()
    
    def drive(self):
        print("Car driving...")
        self.wheels.rotate()

# Create a car with default engine and wheels
car = Car()

# Use the car
car.start()
car.drive()
car.stop()

# Create a car with custom engine and wheels
sports_car = Car(engine_power=400, wheel_count=4)

# Use the sports car
sports_car.start()
sports_car.drive()
sports_car.stop()

In this example, the `Car` class creates its own `Engine` and `Wheels` using factory methods. This approach gives the `Car` class more control over the creation of its components, while still maintaining the benefits of composition.

### Composition with Dependency Injection

Dependency injection is a technique where the dependencies of a class are "injected" from outside, rather than created within the class. This makes the class more flexible and easier to test:

In [ ]:
class Car:
    def __init__(self, engine=None, wheels=None):
        self.engine = engine or Engine(power=150)
        self.wheels = wheels or Wheels(count=4)
    
    def start(self):
        self.engine.start()
    
    def stop(self):
        self.engine.stop()
        self.wheels.stop()
    
    def drive(self):
        print("Car driving...")
        self.wheels.rotate()

# Create a car with default engine and wheels
car = Car()

# Use the car
car.start()
car.drive()
car.stop()

# Create a car with custom engine and wheels
custom_engine = Engine(power=300)
custom_wheels = Wheels(count=6)
custom_car = Car(engine=custom_engine, wheels=custom_wheels)

# Use the custom car
custom_car.start()
custom_car.drive()
custom_car.stop()

In this example, the `Car` class accepts optional `engine` and `wheels` parameters, with default values if they're not provided. This allows for more flexibility in creating cars with different components, while still providing sensible defaults.

## 5. Delegation Pattern <a name="delegation-pattern"></a>

The delegation pattern is a design pattern in which an object, instead of performing a task itself, delegates the task to an associated helper object. This pattern is a fundamental concept in composition.

In Python, delegation can be implemented in several ways:

### Manual Delegation

The most straightforward way to implement delegation is to manually forward method calls to the delegate object:

In [ ]:
class ImageLoader:
    def load(self, path):
        """Load an image from a file."""
        print(f"Loading image from {path}...")
        return cv2.imread(path)

class ImageSaver:
    def save(self, image, path):
        """Save an image to a file."""
        print(f"Saving image to {path}...")
        cv2.imwrite(path, image)

class ImageProcessor:
    def __init__(self, loader=None, saver=None):
        self.loader = loader or ImageLoader()
        self.saver = saver or ImageSaver()
    
    def process(self, image):
        """Process an image (base implementation)."""
        return image
    
    def load_and_process(self, path):
        """Load an image, process it, and return the result."""
        image = self.loader.load(path)
        return self.process(image)
    
    def process_and_save(self, image, path):
        """Process an image and save the result."""
        processed_image = self.process(image)
        self.saver.save(processed_image, path)
        return processed_image
    
    def load_process_and_save(self, input_path, output_path):
        """Load an image, process it, and save the result."""
        image = self.loader.load(input_path)
        processed_image = self.process(image)
        self.saver.save(processed_image, output_path)
        return processed_image

# Create a custom image processor
class GrayscaleProcessor(ImageProcessor):
    def process(self, image):
        """Convert the image to grayscale."""
        print("Converting image to grayscale...")
        if len(image.shape) == 2 or image.shape[2] == 1:
            return image  # Already grayscale
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Create a grayscale processor
grayscale_processor = GrayscaleProcessor()

# Process the sample image
grayscale_image = grayscale_processor.process(sample_image)
display_image(grayscale_image, "Grayscale Image")

# Save the image to a temporary file and load it back
import tempfile
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as temp_file:
    temp_path = temp_file.name

grayscale_processor.process_and_save(sample_image, temp_path)
loaded_image = grayscale_processor.loader.load(temp_path)
display_image(loaded_image, "Loaded Grayscale Image")

# Clean up the temporary file
import os
os.remove(temp_path)

In this example, the `ImageProcessor` class delegates image loading to an `ImageLoader` and image saving to an `ImageSaver`. This separation of concerns makes the code more modular and easier to maintain.

### Delegation with `__getattr__`

Python's `__getattr__` method can be used to automatically delegate method calls to a delegate object:

In [ ]:
class ImageFilter:
    def apply(self, image):
        """Apply the filter to an image (base implementation)."""
        return image
    
    def get_name(self):
        """Get the name of the filter."""
        return "Base Filter"

class GrayscaleFilter(ImageFilter):
    def apply(self, image):
        """Convert the image to grayscale."""
        if len(image.shape) == 2 or image.shape[2] == 1:
            return image  # Already grayscale
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    def get_name(self):
        """Get the name of the filter."""
        return "Grayscale Filter"

class ImageProcessor:
    def __init__(self, filter):
        self.filter = filter
    
    def process(self, image):
        """Process an image using the filter."""
        return self.filter.apply(image)
    
    def __getattr__(self, name):
        """Delegate attribute access to the filter."""
        return getattr(self.filter, name)

# Create a grayscale processor
grayscale_filter = GrayscaleFilter()
grayscale_processor = ImageProcessor(grayscale_filter)

# Process the sample image
grayscale_image = grayscale_processor.process(sample_image)
display_image(grayscale_image, "Grayscale Image")

# Access a method of the filter through the processor
filter_name = grayscale_processor.get_name()
print(f"Filter name: {filter_name}")

In this example, the `ImageProcessor` class uses `__getattr__` to delegate attribute access to its `filter` attribute. This allows the processor to access methods of the filter directly, without having to define wrapper methods for each one.

### Delegation with Descriptors

Python's descriptor protocol can be used to create more sophisticated delegation patterns:

In [ ]:
class Delegate:
    def __init__(self, delegate_name):
        self.delegate_name = delegate_name
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.delegate_name)

class ImageProcessor:
    def __init__(self, filter, loader=None, saver=None):
        self._filter = filter
        self._loader = loader or ImageLoader()
        self._saver = saver or ImageSaver()
    
    filter = Delegate("_filter")
    loader = Delegate("_loader")
    saver = Delegate("_saver")
    
    def process(self, image):
        """Process an image using the filter."""
        return self.filter.apply(image)
    
    def load_and_process(self, path):
        """Load an image, process it, and return the result."""
        image = self.loader.load(path)
        return self.process(image)
    
    def process_and_save(self, image, path):
        """Process an image and save the result."""
        processed_image = self.process(image)
        self.saver.save(processed_image, path)
        return processed_image

# Create a grayscale processor
grayscale_filter = GrayscaleFilter()
grayscale_processor = ImageProcessor(grayscale_filter)

# Process the sample image
grayscale_image = grayscale_processor.process(sample_image)
display_image(grayscale_image, "Grayscale Image")

# Access the filter directly
filter_name = grayscale_processor.filter.get_name()
print(f"Filter name: {filter_name}")

# Save the image to a temporary file
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as temp_file:
    temp_path = temp_file.name

grayscale_processor.process_and_save(sample_image, temp_path)

# Clean up the temporary file
os.remove(temp_path)

In this example, the `Delegate` descriptor is used to create properties that delegate to internal attributes. This provides a cleaner interface for accessing the delegated objects, while still maintaining encapsulation.

## 6. Composition in Computer Vision <a name="cv-applications"></a>

Composition is particularly useful in computer vision applications, where you often need to combine multiple processing steps or algorithms. Let's see a more comprehensive example of how composition can be applied to create a flexible image processing pipeline.

In [ ]:
class ImageOperation:
    def __init__(self, name="Base Operation"):
        self.name = name
    
    def apply(self, image):
        """Apply the operation to an image (base implementation)."""
        return image

class GrayscaleOperation(ImageOperation):
    def __init__(self):
        super().__init__("Grayscale")
    
    def apply(self, image):
        """Convert the image to grayscale."""
        if len(image.shape) == 2 or image.shape[2] == 1:
            return image  # Already grayscale
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

class BlurOperation(ImageOperation):
    def __init__(self, kernel_size=(5, 5)):
        super().__init__(f"Blur ({kernel_size})")
        self.kernel_size = kernel_size
    
    def apply(self, image):
        """Apply Gaussian blur to the image."""
        return cv2.GaussianBlur(image, self.kernel_size, 0)

class ThresholdOperation(ImageOperation):
    def __init__(self, threshold=127, max_value=255, threshold_type=cv2.THRESH_BINARY):
        super().__init__(f"Threshold ({threshold}, {max_value}, {threshold_type})")
        self.threshold = threshold
        self.max_value = max_value
        self.threshold_type = threshold_type
    
    def apply(self, image):
        """Apply thresholding to the image."""
        # Convert to grayscale if needed
        if len(image.shape) == 3 and image.shape[2] > 1:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image
        
        _, result = cv2.threshold(gray, self.threshold, self.max_value, self.threshold_type)
        return result

class EdgeDetectionOperation(ImageOperation):
    def __init__(self, threshold1=100, threshold2=200):
        super().__init__(f"Edge Detection ({threshold1}, {threshold2})")
        self.threshold1 = threshold1
        self.threshold2 = threshold2
    
    def apply(self, image):
        """Detect edges in the image using Canny edge detection."""
        # Convert to grayscale if needed
        if len(image.shape) == 3 and image.shape[2] > 1:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image
        
        return cv2.Canny(gray, self.threshold1, self.threshold2)

class MorphologyOperation(ImageOperation):
    def __init__(self, operation=cv2.MORPH_OPEN, kernel_size=(5, 5)):
        operation_name = {
            cv2.MORPH_OPEN: "OPEN",
            cv2.MORPH_CLOSE: "CLOSE",
            cv2.MORPH_GRADIENT: "GRADIENT",
            cv2.MORPH_TOPHAT: "TOPHAT",
            cv2.MORPH_BLACKHAT: "BLACKHAT",
            cv2.MORPH_ERODE: "ERODE",
            cv2.MORPH_DILATE: "DILATE"
        }.get(operation, str(operation))
        
        super().__init__(f"Morphology {operation_name} ({kernel_size})")
        self.operation = operation
        self.kernel_size = kernel_size
        self.kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    
    def apply(self, image):
        """Apply morphological operation to the image."""
        return cv2.morphologyEx(image, self.operation, self.kernel)

class ImagePipeline:
    def __init__(self, operations=None, name="Image Pipeline"):
        self.operations = operations or []
        self.name = name
    
    def add_operation(self, operation):
        """Add an operation to the pipeline."""
        self.operations.append(operation)
        return self  # For method chaining
    
    def process(self, image):
        """Process an image through the pipeline."""
        result = image.copy()
        
        for operation in self.operations:
            result = operation.apply(result)
        
        return result
    
    def visualize_steps(self, image):
        """Process an image through the pipeline and visualize each step."""
        results = [image.copy()]
        titles = ["Original"]
        
        result = image.copy()
        for operation in self.operations:
            result = operation.apply(result)
            results.append(result.copy())
            titles.append(operation.name)
        
        # Determine the number of rows and columns for the subplot grid
        n_images = len(results)
        n_cols = min(3, n_images)  # Maximum 3 columns
        n_rows = (n_images + n_cols - 1) // n_cols  # Ceiling division
        
        plt.figure(figsize=(15, 5 * n_rows))
        
        for i, (result, title) in enumerate(zip(results, titles)):
            plt.subplot(n_rows, n_cols, i + 1)
            if len(result.shape) == 3:  # Color image
                plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            else:  # Grayscale image
                plt.imshow(result, cmap='gray')
            plt.title(title)
            plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return results[-1]  # Return the final result

# Create some pipelines
edge_detection_pipeline = ImagePipeline(name="Edge Detection Pipeline")
edge_detection_pipeline.add_operation(GrayscaleOperation())
edge_detection_pipeline.add_operation(BlurOperation(kernel_size=(5, 5)))
edge_detection_pipeline.add_operation(EdgeDetectionOperation(threshold1=50, threshold2=150))

threshold_pipeline = ImagePipeline(name="Threshold Pipeline")
threshold_pipeline.add_operation(GrayscaleOperation())
threshold_pipeline.add_operation(BlurOperation(kernel_size=(5, 5)))
threshold_pipeline.add_operation(ThresholdOperation(threshold=100, threshold_type=cv2.THRESH_BINARY))

morphology_pipeline = ImagePipeline(name="Morphology Pipeline")
morphology_pipeline.add_operation(GrayscaleOperation())
morphology_pipeline.add_operation(ThresholdOperation(threshold=100, threshold_type=cv2.THRESH_BINARY))
morphology_pipeline.add_operation(MorphologyOperation(operation=cv2.MORPH_OPEN, kernel_size=(5, 5)))

# Process and visualize the results
edge_detection_pipeline.visualize_steps(sample_image)
threshold_pipeline.visualize_steps(sample_image)
morphology_pipeline.visualize_steps(sample_image)

In this example, we've created a flexible image processing pipeline using composition. The `ImagePipeline` class is composed of a list of `ImageOperation` objects, each of which can apply a specific operation to an image. This allows us to create different pipelines with different combinations of operations, all using the same interface.

This is a powerful application of composition in computer vision. It allows us to create modular, reusable components that can be combined in various ways to achieve different image processing tasks.

Key benefits of this approach:

1. **Modularity**: Each operation is a self-contained module that can be used independently or as part of a pipeline.
2. **Reusability**: Operations can be reused in different pipelines.
3. **Extensibility**: New operations can be added without changing existing code.
4. **Flexibility**: Pipelines can be configured with different combinations of operations.
5. **Maintainability**: Each operation has a single responsibility, making the code easier to maintain.

## 7. Best Practices <a name="best-practices"></a>

Here are some best practices for using composition in Python:

### Designing Composable Classes

1. **Single Responsibility Principle**: Each class should have a single responsibility, making it easier to compose classes with different responsibilities.
2. **Interface Segregation**: Define small, focused interfaces rather than large, monolithic ones, making it easier to compose classes that implement these interfaces.
3. **Dependency Inversion**: Depend on abstractions, not concrete implementations, making it easier to swap out components.
4. **Favor Composition Over Inheritance**: Use composition for "has-a" relationships and inheritance for "is-a" relationships.
5. **Design for Testability**: Make it easy to test classes in isolation by allowing dependencies to be injected.

### Implementing Composition

1. **Use Dependency Injection**: Pass dependencies to a class rather than creating them within the class, making the class more flexible and easier to test.
2. **Provide Sensible Defaults**: Provide default implementations for dependencies, making the class easier to use while still allowing for customization.
3. **Use Factory Methods**: Use factory methods to create complex objects, encapsulating the creation logic.
4. **Consider Delegation**: Use delegation to forward method calls to composed objects, avoiding code duplication.
5. **Document Composition Relationships**: Clearly document the composition relationships between classes, making the code easier to understand.

### Using Composition

1. **Favor Object Composition**: Use object composition to achieve code reuse and flexibility.
2. **Compose at Runtime**: Take advantage of the ability to change composition at runtime for more dynamic behavior.
3. **Use Composition for Variation**: Use composition to handle variations in behavior, rather than creating deep inheritance hierarchies.
4. **Balance Composition and Inheritance**: Use both composition and inheritance where appropriate, rather than dogmatically favoring one over the other.
5. **Refactor to Composition**: Consider refactoring inheritance hierarchies to use composition when they become unwieldy or inflexible.

## 8. Exercises <a name="exercises"></a>

Now that you've learned about composition, try these exercises to reinforce your understanding:

### Exercise 1: Create a Camera System

Create a camera system with the following requirements:

1. Create a `Camera` class that is composed of a `Lens`, a `Sensor`, and a `Processor`.
2. Each component should have properties that affect the final image (e.g., the lens affects focus, the sensor affects resolution, the processor affects color).
3. The `Camera` should have a method to capture an image, which delegates to its components.
4. Create different types of lenses, sensors, and processors, and experiment with different combinations.
5. Implement a method to change components at runtime (e.g., change the lens or processor).

### Exercise 2: Create a Feature Extraction Pipeline

Create a feature extraction pipeline with the following requirements:

1. Create a `FeatureExtractor` class that is composed of a `Preprocessor`, a `Detector`, and a `Descriptor`.
2. Each component should have a specific role in the feature extraction process (e.g., the preprocessor prepares the image, the detector finds keypoints, the descriptor computes feature vectors).
3. The `FeatureExtractor` should have a method to extract features from an image, which delegates to its components.
4. Create different types of preprocessors, detectors, and descriptors, and experiment with different combinations.
5. Implement a method to visualize the features extracted by the pipeline.

### Exercise 3: Create an Image Classification System

Create an image classification system with the following requirements:

1. Create a `Classifier` class that is composed of a `FeatureExtractor` and a `Model`.
2. The `FeatureExtractor` should extract features from an image, and the `Model` should classify the features.
3. The `Classifier` should have a method to classify an image, which delegates to its components.
4. Create different types of feature extractors and models, and experiment with different combinations.
5. Implement a method to evaluate the performance of the classifier on a dataset.

### Exercise 4: Create an Image Processing Application

Create an image processing application with the following requirements:

1. Create an `Application` class that is composed of an `ImageLoader`, an `ImageProcessor`, and an `ImageSaver`.
2. The application should allow the user to load an image, apply various processing operations, and save the result.
3. The `ImageProcessor` should be composed of a list of `Operation` objects, each of which applies a specific operation to an image.
4. Create different types of operations (e.g., filters, transformations, enhancements), and allow the user to add them to the processor.
5. Implement a method to visualize the effect of each operation in the processing pipeline.

## 9. Conclusion <a name="conclusion"></a>

In this notebook, we've explored the principle of composition over inheritance, a powerful design approach that favors object composition over class inheritance. We've seen how this principle can be applied to create more flexible, maintainable, and realistic models for computer vision applications.

Key takeaways:

1. **Composition over inheritance** is a design principle that suggests that classes should achieve polymorphic behavior and code reuse by their composition rather than through inheritance.

2. Composition represents a "has-a" relationship between classes, while inheritance represents an "is-a" relationship.

3. Composition offers several advantages over inheritance, including greater flexibility, reduced coupling, and avoidance of the fragile base class problem.

4. Python makes it easy to implement composition through its flexible object model, with patterns such as basic composition, composition with factory methods, and composition with dependency injection.

5. The delegation pattern is a fundamental concept in composition, where an object delegates tasks to associated helper objects.

6. Composition is particularly useful in computer vision applications, where you often need to combine multiple processing steps or algorithms.

7. Best practices for using composition include following the Single Responsibility Principle, using dependency injection, providing sensible defaults, and documenting composition relationships.

By understanding and applying composition effectively, you can create more flexible, extensible, and maintainable code for your computer vision applications.